# Reconhecimento de padrões com redes siamesas

## Estudo acadêmico: comparação de imagens com transferência de aprendizado

**Autor:** Gabriel Passos de Jesus  
**Natureza:** atividade acadêmica desenvolvida em contexto de disciplina  
**Ambiente de referência:** Google Colab + TensorFlow/Keras  
**Base de dados:** Fruits-360

---

### Resumo

Este notebook apresenta um experimento didático de reconhecimento de padrões por comparação de pares de imagens. A abordagem utiliza uma **rede siamesa (Siamese Neural Network)**: duas entradas compartilham o mesmo extrator de características, e uma camada de comparação estima se as imagens pertencem à mesma categoria. O extrator é inicializado com pesos pré-treinados do VGG16 (ImageNet), caracterizando uma estratégia de transferência de aprendizado.

O objetivo é estudar o fluxo metodológico — preparação dos dados, formação de pares positivos e negativos, treinamento e avaliação *N-way one-shot*. Os resultados gerados durante uma execução devem ser interpretados como resultados do experimento naquela configuração, não como benchmark definitivo.

> **Importante:** este projeto é uma atividade acadêmica/educacional. Não constitui sistema de classificação pronto para produção nem estudo experimental com validação estatística abrangente.


## 1. Objetivos

### Objetivo geral
Implementar e analisar uma arquitetura siamesa para aprender uma representação de imagens que permita comparar a similaridade entre pares.

### Objetivos específicos
- Preparar imagens e rótulos a partir de diretórios organizados por categoria.
- Separar categorias de treinamento e avaliação, evitando que uma mesma classe apareça nos dois conjuntos.
- Construir pares positivos (mesma categoria) e negativos (categorias distintas).
- Utilizar o VGG16 pré-treinado como extrator compartilhado de características.
- Treinar a rede com entropia cruzada binária.
- Avaliar a capacidade de comparação em uma tarefa *N-way one-shot*.
- Discutir limitações, reprodutibilidade e possibilidades de evolução.

## 2. Fundamentos metodológicos

Uma rede siamesa processa duas entradas por meio de uma mesma função de codificação, com pesos compartilhados. As representações resultantes são comparadas — neste estudo, pela diferença absoluta elemento a elemento — e encaminhadas a uma camada sigmoide que estima a probabilidade de as imagens pertencerem à mesma categoria.

O treinamento é realizado com pares:
- **Positivos (rótulo 1):** duas imagens da mesma categoria.
- **Negativos (rótulo 0):** imagens de categorias diferentes.

A avaliação *N-way one-shot* apresenta uma imagem de consulta e N imagens candidatas. Apenas uma candidata pertence à mesma categoria da consulta; a rede deve atribuir a maior similaridade à candidata correta.


## 3. Ambiente e dependências

O notebook foi organizado para execução no Google Colab. O TensorFlow/Keras é utilizado para a arquitetura e o treinamento; NumPy e Matplotlib apoiam o processamento e a visualização.

A primeira execução do VGG16 com `weights="imagenet"` pode exigir acesso à internet para baixar os pesos pré-treinados.

**Estrutura esperada dos dados:**

```text
Training/
├── Apple/
│   ├── image_001.jpg
│   └── ...
├── Banana/
│   └── ...
└── ...
```

O diretório de cada categoria deve conter imagens válidas. A base Fruits-360 não é distribuída neste repositório; obtenha-a na fonte oficial e configure o caminho abaixo.


## 4. Configuração e reprodutibilidade

A separação é feita **por categoria**, e não por imagem: categorias destinadas à avaliação não são utilizadas na formação dos pares de treinamento. Essa escolha permite investigar a comparação em categorias não vistas durante o treinamento.

A quantidade de imagens por categoria é limitada por `SAMPLES_PER_CLASS` para manter o experimento didático e computacionalmente acessível. Para uma análise mais robusta, aumente esse valor e repita o experimento com diferentes sementes.


In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, Lambda
)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.preprocessing.image import load_img, img_to_array

SEED = 42
IMAGE_SIZE = (224, 224)
SAMPLES_PER_CLASS = 5
TRAIN_CLASS_FRACTION = 0.70
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
N_WAY = 10
N_EVALUATION_TASKS = 100

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponível:", bool(tf.config.list_physical_devices("GPU")))

### 4.1. Acesso à base de dados

No Google Colab, monte o Google Drive e ajuste `BASE_DIR` para a pasta `Training` da Fruits-360. Se estiver executando em outro ambiente, informe um caminho local válido.



In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/data/fruits-360/Training")

if not BASE_DIR.is_dir():
    raise FileNotFoundError(
        f"Diretório não encontrado: {BASE_DIR}\n"
        "Ajuste BASE_DIR para a pasta Training da Fruits-360."
    )

print("Diretório de dados:", BASE_DIR)

## 5. Carregamento e preparação das imagens

As imagens são redimensionadas para 224 × 224 pixels, convertidas para RGB e armazenadas como valores em ponto flutuante. O pré-processamento específico do VGG16 é aplicado posteriormente, preservando a escala esperada pelo modelo pré-treinado.

A leitura é determinística: categorias e arquivos são ordenados e a seleção de amostras é controlada pela semente definida anteriormente.


In [ ]:
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

class_names = sorted(
    p.name for p in BASE_DIR.iterdir()
    if p.is_dir()
)

if len(class_names) < 2:
    raise ValueError("A base precisa conter pelo menos duas categorias.")

images_by_class = {}
for class_name in class_names:
    files = sorted(
        p for p in (BASE_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
    )
    selected = files[:SAMPLES_PER_CLASS]
    if len(selected) < 2:
        continue

    class_images = []
    for image_path in selected:
        image = load_img(image_path, target_size=IMAGE_SIZE, color_mode="rgb")
        array = img_to_array(image).astype("float32")
        class_images.append(array)

    images_by_class[class_name] = np.stack(class_images)

class_names = sorted(images_by_class)
if len(class_names) < 2:
    raise ValueError(
        "Não há pelo menos duas categorias com duas imagens válidas cada."
    )

print(f"Categorias carregadas: {len(class_names)}")
print(f"Imagens por categoria: até {SAMPLES_PER_CLASS}")
print("Exemplo de formato:", images_by_class[class_names[0]].shape)

### 5.1. Inspeção visual

In [ ]:
fig, axes = plt.subplots(1, min(5, len(class_names)), figsize=(15, 3))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for ax, class_name in zip(axes, class_names[:len(axes)]):
    ax.imshow(images_by_class[class_name][0].astype("uint8"))
    ax.set_title(class_name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Separação entre treinamento e avaliação

A divisão ocorre no nível das categorias. Uma parte das categorias será utilizada para treinamento e as categorias restantes serão reservadas para a avaliação *N-way one-shot*.

Essa estratégia evita a sobreposição de categorias entre os dois conjuntos. Ainda assim, por se tratar de uma amostra pequena e de uma única base, a avaliação não permite generalizações amplas sobre outros domínios de imagem.


In [ ]:
rng = np.random.default_rng(SEED)
shuffled_classes = np.array(class_names, dtype=object)
rng.shuffle(shuffled_classes)

n_train_classes = int(len(shuffled_classes) * TRAIN_CLASS_FRACTION)
n_train_classes = min(max(n_train_classes, 1), len(shuffled_classes) - 1)

train_classes = list(shuffled_classes[:n_train_classes])
evaluation_classes = list(shuffled_classes[n_train_classes:])

train_images_by_class = {k: images_by_class[k] for k in train_classes}
eval_images_by_class = {k: images_by_class[k] for k in evaluation_classes}

print("Categorias de treinamento:", len(train_classes))
print("Categorias de avaliação:", len(evaluation_classes))
print("Interseção entre conjuntos:", set(train_classes) & set(evaluation_classes))

assert set(train_classes).isdisjoint(set(evaluation_classes))

## 7. Formação de pares para treinamento

Cada lote contém pares positivos e negativos em proporções equilibradas. Para cada par, a primeira imagem é selecionada de uma categoria de treinamento. A segunda imagem é selecionada da mesma categoria (positivo) ou de outra categoria (negativo).

O rótulo 1 representa mesma categoria; o rótulo 0 representa categorias diferentes.


In [ ]:
def make_training_batch(batch_size=BATCH_SIZE):
    half = batch_size // 2
    labels = np.array([1] * half + [0] * (batch_size - half), dtype="float32")
    rng.shuffle(labels)

    left_batch = np.empty((batch_size, *IMAGE_SIZE, 3), dtype="float32")
    right_batch = np.empty_like(left_batch)

    for i, label in enumerate(labels):
        class_name = random.choice(train_classes)
        left_images = train_images_by_class[class_name]
        left_batch[i] = left_images[random.randrange(len(left_images))]

        if label == 1:
            # Par positivo: imagem distinta, da mesma categoria.
            idx_left = random.randrange(len(left_images))
            idx_right = (idx_left + random.randrange(1, len(left_images))) % len(left_images)
            left_batch[i] = left_images[idx_left]
            right_batch[i] = left_images[idx_right]
        else:
            negative_class = random.choice(
                [name for name in train_classes if name != class_name]
            )
            right_images = train_images_by_class[negative_class]
            right_batch[i] = right_images[random.randrange(len(right_images))]

    left_batch = preprocess_input(left_batch.copy())
    right_batch = preprocess_input(right_batch.copy())
    return [left_batch, right_batch], labels.reshape(-1, 1)

sample_pairs, sample_labels = make_training_batch()
print("Formato das entradas:", sample_pairs[0].shape, sample_pairs[1].shape)
print("Formato dos rótulos:", sample_labels.shape)
print("Proporção positiva:", float(sample_labels.mean()))

## 8. Arquitetura da rede siamesa

O VGG16 é carregado sem as camadas de classificação originais e com pesos do ImageNet. Seus pesos são congelados nesta versão didática. Uma camada de *Global Average Pooling* reduz a representação espacial; uma camada densa gera o vetor de características.

O mesmo codificador é aplicado às duas entradas, garantindo compartilhamento de pesos. A diferença absoluta entre os vetores é processada por uma camada sigmoide, que retorna uma pontuação de similaridade entre 0 e 1.

**Observação:** nesta configuração, o VGG16 permanece congelado. O treinamento ajusta as camadas densas adicionadas ao codificador e a camada de comparação. O ajuste fino (*fine-tuning*) do backbone pode ser explorado posteriormente.


In [ ]:
def build_siamese_network():
    backbone = VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMAGE_SIZE, 3)
    )
    backbone.trainable = False

    encoder_input = Input(shape=(*IMAGE_SIZE, 3), name="image")
    features = backbone(encoder_input, training=False)
    features = GlobalAveragePooling2D(name="global_average_pooling")(features)
    features = Dense(256, activation="relu", name="embedding")(features)
    features = Dropout(0.2, name="embedding_dropout")(features)
    encoder = Model(encoder_input, features, name="shared_encoder")

    left_input = Input(shape=(*IMAGE_SIZE, 3), name="left_image")
    right_input = Input(shape=(*IMAGE_SIZE, 3), name="right_image")

    left_embedding = encoder(left_input)
    right_embedding = encoder(right_input)

    distance = Lambda(
        lambda tensors: tf.abs(tensors[0] - tensors[1]),
        name="absolute_difference"
    )([left_embedding, right_embedding])

    similarity = Dense(1, activation="sigmoid", name="similarity")(distance)
    network = Model(
        inputs=[left_input, right_input],
        outputs=similarity,
        name="siamese_vgg16"
    )

    network.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=BinaryCrossentropy(),
        metrics=["accuracy"]
    )
    return network

siamese_net = build_siamese_network()
siamese_net.summary()

### 8.1. Teste de integridade antes do treinamento

Antes de iniciar as épocas, executamos verificações rápidas para confirmar as dimensões dos lotes, o balanceamento dos rótulos e a saída da arquitetura. Esses testes são verificações funcionais básicas; não substituem a execução integral nem uma avaliação científica independente.

In [ ]:
# Teste do gerador de pares
test_batch_x, test_batch_y = make_training_batch(batch_size=8)

assert len(test_batch_x) == 2
assert test_batch_x[0].shape == (8, *IMAGE_SIZE, 3)
assert test_batch_x[1].shape == (8, *IMAGE_SIZE, 3)
assert test_batch_y.shape == (8, 1)
assert set(np.unique(test_batch_y)).issubset({0.0, 1.0})
assert int(test_batch_y.sum()) == 4, "O lote de teste deve conter pares equilibrados."

print("OK: gerador de pares")
print("Entradas:", test_batch_x[0].shape, test_batch_x[1].shape)
print("Rótulos:", test_batch_y.ravel())

In [ ]:
siamese_net = build_siamese_network()

# Teste de formato e faixa da predição
test_prediction = siamese_net.predict(test_batch_x, verbose=0)

assert test_prediction.shape == (8, 1)
assert np.all(np.isfinite(test_prediction))
assert np.all((test_prediction >= 0) & (test_prediction <= 1))

print("OK: arquitetura siamesa")
print("Formato da saída:", test_prediction.shape)
print("Faixa das predições:", float(test_prediction.min()), "a", float(test_prediction.max()))
siamese_net.summary()

## 9. Treinamento

O treinamento utiliza lotes gerados sob demanda, evitando materializar antecipadamente todos os pares possíveis. A cada época, um novo conjunto de pares é amostrado.

Os valores de perda e acurácia abaixo são registrados durante a execução. Eles não devem ser confundidos com resultados previamente obtidos ou com uma validação independente.


In [ ]:
steps_per_epoch = max(10, min(50, len(train_classes)))
history = {"loss": [], "accuracy": []}
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_losses = []
    epoch_accuracies = []

    for _ in range(steps_per_epoch):
        batch_x, batch_y = make_training_batch()
        metrics = siamese_net.train_on_batch(batch_x, batch_y, return_dict=True)
        epoch_losses.append(float(metrics["loss"]))
        epoch_accuracies.append(float(metrics["accuracy"]))

    mean_loss = float(np.mean(epoch_losses))
    mean_accuracy = float(np.mean(epoch_accuracies))
    history["loss"].append(mean_loss)
    history["accuracy"].append(mean_accuracy)

    print(
        f"Época {epoch + 1:02d}/{EPOCHS} | "
        f"loss={mean_loss:.4f} | accuracy={mean_accuracy:.4f}"
    )

training_seconds = time.time() - start_time
print(f"Tempo de treinamento: {training_seconds:.1f} s")

### 9.1. Curvas de treinamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["loss"])
axes[0].set_title("Perda por época")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Binary cross-entropy")

axes[1].plot(history["accuracy"])
axes[1].set_title("Acurácia nos lotes de treinamento")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Acurácia")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 10. Avaliação N-way one-shot

Cada tarefa seleciona uma categoria de avaliação, uma imagem de consulta e N imagens candidatas. Uma candidata é da mesma categoria da consulta; as demais pertencem a categorias diferentes. A predição é considerada correta quando a candidata positiva recebe a maior pontuação de similaridade.

A avaliação é repetida em várias tarefas amostradas aleatoriamente. A acurácia reportada é a proporção de tarefas em que a candidata correta ficou em primeiro lugar.


In [ ]:
def evaluate_n_way(model, n_way=N_WAY, n_tasks=N_EVALUATION_TASKS):
    if len(evaluation_classes) < n_way:
        raise ValueError(
            f"N-way={n_way} requer ao menos {n_way} categorias de avaliação; "
            f"há {len(evaluation_classes)}."
        )

    correct = 0
    scores_all = []

    for _ in range(n_tasks):
        target_class = random.choice(evaluation_classes)
        target_images = eval_images_by_class[target_class]

        query_idx, positive_idx = random.sample(range(len(target_images)), 2)
        query = target_images[query_idx]

        distractor_classes = random.sample(
            [name for name in evaluation_classes if name != target_class],
            n_way - 1
        )

        candidates = [target_images[positive_idx]]
        for class_name in distractor_classes:
            candidate_images = eval_images_by_class[class_name]
            candidates.append(candidate_images[random.randrange(len(candidate_images))])

        # A posição da candidata positiva é aleatória.
        positive_position = random.randrange(n_way)
        candidates[0], candidates[positive_position] = (
            candidates[positive_position], candidates[0]
        )

        left = np.repeat(query[np.newaxis, ...], n_way, axis=0)
        right = np.stack(candidates)

        left = preprocess_input(left.copy())
        right = preprocess_input(right.copy())

        scores = model.predict([left, right], verbose=0).reshape(-1)
        predicted_position = int(np.argmax(scores))
        correct += int(predicted_position == positive_position)
        scores_all.append(float(scores[positive_position]))

    accuracy = correct / n_tasks
    return {
        "accuracy": accuracy,
        "correct": correct,
        "tasks": n_tasks,
        "mean_positive_score": float(np.mean(scores_all))
    }

evaluation = evaluate_n_way(siamese_net)
print(f"Acurácia N-way: {evaluation['accuracy']:.3f}")
print(f"Tarefas corretas: {evaluation['correct']} / {evaluation['tasks']}")
print(f"Similaridade média do par positivo: {evaluation['mean_positive_score']:.4f}")

## 11. Interpretação e limitações

A avaliação *N-way one-shot* mede a capacidade relativa da rede de selecionar, entre N candidatas, aquela que corresponde à categoria da consulta. A acurácia depende do valor de N, da amostragem das tarefas, do número de imagens por categoria e da diversidade da base.

### Limitações desta implementação
1. **Amostragem reduzida:** utilizar poucas imagens por categoria torna o experimento mais rápido, mas limita a diversidade visual.
2. **Uma única base:** o resultado não demonstra generalização para outras bases, modalidades ou domínios.
3. **Backbone congelado:** o VGG16 não é ajustado ao domínio Fruits-360 nesta versão.
4. **Amostragem estocástica:** diferentes sementes e tarefas podem produzir resultados distintos.
5. **Sem conjunto de validação separado:** a avaliação é feita em categorias reservadas, mas não há uma etapa independente de seleção de hiperparâmetros.
6. **Sem comparação com baselines:** não foram incluídos classificadores convencionais, embeddings sem treinamento ou outras arquiteturas.
7. **Sem intervalo de confiança:** uma análise mais rigorosa deve repetir o experimento com múltiplas sementes e reportar dispersão/intervalos de confiança.

### Cuidados para uma versão de pesquisa
- Aumentar o número de imagens por classe e documentar a versão exata da base.
- Executar múltiplas sementes e reportar média e desvio-padrão.
- Comparar diferentes valores de N e diferentes estratégias de amostragem.
- Avaliar precisão, revocação, F1, ROC-AUC e matriz de confusão em uma tarefa binária definida sem vazamento.
- Registrar versões das bibliotecas, hardware, hiperparâmetros e tempo de execução.
- Incluir baselines e análise qualitativa de erros.


## 12. Salvamento do modelo

O modelo completo pode ser salvo no formato Keras. O arquivo gerado não é versionado automaticamente neste repositório; salve-o no ambiente de execução ou no Google Drive conforme a necessidade.


In [ ]:
MODEL_PATH = "/content/siamese_fruits360.keras"
siamese_net.save(MODEL_PATH)
print("Modelo salvo em:", MODEL_PATH)

## 13. Conclusão

O exercício demonstra como redes siamesas podem aprender uma função de comparação entre imagens, em vez de produzir diretamente uma classe entre um conjunto fixo de rótulos. A combinação de pesos compartilhados, representação vetorial e treinamento com pares positivos e negativos oferece uma introdução prática a tarefas de similaridade e aprendizado *one-shot*.

O valor principal deste repositório é didático: tornar explícitas as etapas de preparação, modelagem, treinamento e avaliação, além de registrar as limitações que precisam ser tratadas antes de qualquer afirmação de desempenho generalizável.

## 14. Referências

- Keras / TensorFlow. **VGG16** — documentação da arquitetura e dos pesos pré-treinados: https://www.tensorflow.org/api_docs/python/tf/keras/applications/VGG16
- Keras / TensorFlow. **VGG16 preprocess_input** — pré-processamento de imagens para VGG16: https://www.tensorflow.org/api_docs/python/tf/keras/applications/vgg16/preprocess_input
- Koch, G., Zemel, R., & Salakhutdinov, R. (2015). **Siamese Neural Networks for One-shot Image Recognition.**  
- Material introdutório utilizado como ponto de partida para a atividade: https://towardsdatascience.com/siamese-networks-line-by-line-explanation-for-beginners-55b8be1d2fc6

---

**Autoria:** Gabriel Passos de Jesus  
**GitHub:** https://github.com/iamdatagabs  
**Repositório:** https://github.com/iamdatagabs/reconhecimentopadroes  
**Portfólio:** https://iamdatagabs.github.io/iamdatagabs/portfolio/

*Atividade acadêmica com finalidade educacional. A implementação deve ser executada e reavaliada no ambiente do usuário antes de qualquer uso de seus resultados como evidência experimental.*